# Mini Semantic Search

## 1. Dataset Inspection

In this section, we will inspect the raw Video Games Question and Answer dataset before performing any data cleaning. Understanding the file structure and the information it contains will help us choose the appropriate methods for loading and preparing the data.

The following cell uses Python's built-in file handling to read the beginning of the JSON file. We will inspect a small portion of the data to understand its structure without loading the entire file into memory.

In [10]:
from pathlib import Path

file_path = Path("../data/QA_Video_Games.json")

with file_path.open("r", encoding="utf-8") as file:
    for _ in range(3):
        print(file.readline())

{'asin': 'B0000512IE', 'questions': [{'questionType': 'open-ended', 'askerID': u'A39TQ7XW6P36BU', 'questionTime': u'June 29, 2014', 'questionText': u'Can I use these with Window 8/8.1?', 'answers': [{'answerText': u"Yes, you will need to go to their website to download version 4.55 or above of the CH Control Manager program, then you'll be all set. Find it here: http://www.chproducts.com/13-29170-Downloads-and-Community.php", 'answererID': u'AH14F9TYDABNH', 'answerTime': u'June 29, 2014', 'helpful': [2, 2]}, {'answerText': u'Sure can. My Windows 8.1 machine automatically detected the appropriate drivers and installed them for me.', 'answererID': u'A36BQQD67VJOD2', 'answerTime': u'June 29, 2014', 'helpful': [2, 2]}, {'answerText': u"Mine work with W8, can't answer for W8.1 but since it does not require drivers or cd's to be recognized or initialized, I believe it may work. It is truly plug and play!", 'answererID': u'AZY7TFN31JV8M', 'answerTime': u'June 29, 2014', 'helpful': [1, 1]}, {'

### 1.1 Inspecting the Raw Dataset

The dataset contains product identifiers and nested question-and-answer records. Before loading the data into Pandas, we need to inspect its serialization format and understand how its records are organized.

In this step, we will read a small sample of the file to determine how the records are stored. This will help us select an appropriate parsing method without loading the entire dataset into memory.

The following code reads the first line of the dataset and prints a short preview. We use Python's built-in file handling so we can inspect the data without installing additional libraries.

In [11]:
from pathlib import Path

file_path = Path("../data/QA_Video_Games.json")

with file_path.open("r", encoding="utf-8") as file:
    first_line = file.readline()

print("First 1000 characters:")
print(first_line[:1000])

print("\nTotal characters in first line:", len(first_line))

First 1000 characters:
{'asin': 'B0000512IE', 'questions': [{'questionType': 'open-ended', 'askerID': u'A39TQ7XW6P36BU', 'questionTime': u'June 29, 2014', 'questionText': u'Can I use these with Window 8/8.1?', 'answers': [{'answerText': u"Yes, you will need to go to their website to download version 4.55 or above of the CH Control Manager program, then you'll be all set. Find it here: http://www.chproducts.com/13-29170-Downloads-and-Community.php", 'answererID': u'AH14F9TYDABNH', 'answerTime': u'June 29, 2014', 'helpful': [2, 2]}, {'answerText': u'Sure can. My Windows 8.1 machine automatically detected the appropriate drivers and installed them for me.', 'answererID': u'A36BQQD67VJOD2', 'answerTime': u'June 29, 2014', 'helpful': [2, 2]}, {'answerText': u"Mine work with W8, can't answer for W8.1 but since it does not require drivers or cd's to be recognized or initialized, I believe it may work. It is truly plug and play!", 'answererID': u'AZY7TFN31JV8M', 'answerTime': u'June 29, 2014',

### 1.2 Parsing a Sample Record

The raw dataset uses Python-style dictionary representations rather than standard JSON. We will use Python's `ast.literal_eval()` function to parse the first record into a Python dictionary.

This allows us to inspect the nested structure without modifying the original file. We will also examine the available fields and count the questions and answers associated with the first product.

In [12]:
import ast

# Convert the first line into a Python dictionary
record = ast.literal_eval(first_line)

# Inspect the record's structure
print("Record type:", type(record))
print("Available fields:", list(record.keys()))

print("\nProduct ID:", record["asin"])
print("Number of questions:", len(record["questions"]))

# Count all answers associated with this product
answer_count = sum(
    len(question.get("answers", []))
    for question in record["questions"]
)

print("Number of answers:", answer_count)

Record type: <class 'dict'>
Available fields: ['asin', 'questions']

Product ID: B0000512IE
Number of questions: 9
Number of answers: 36


### 1.3 Counting Records in the Raw Dataset

Before loading the dataset into Pandas, we need to understand its size. The following code counts the number of non-empty lines in the file. Since our initial inspection suggests that each line represents one product record, this gives us an estimate of the number of product records available for processing.

In [13]:
with file_path.open("r", encoding="utf-8") as file:
    total_records = sum(1 for line in file if line.strip())

print("Total product records:", total_records)

Total product records: 1183


### 1.4 Loading and Validating the Dataset

The dataset contains product questions and their answers. Before using Pandas to analyze the data, we will attempt to parse every non-empty line into a Python dictionary.

We will use `ast.literal_eval()` because our initial inspection suggests that the file contains Python-style dictionary representations rather than standard JSON. We will also count any records that fail to parse so that we can identify potential data quality issues.

In [14]:
import ast

records = []
failed_records = 0

with file_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        try:
            record = ast.literal_eval(line)
            records.append(record)
        except (ValueError, SyntaxError):
            failed_records += 1

print("Successfully loaded records:", len(records))
print("Failed records:", failed_records)

Successfully loaded records: 1183
Failed records: 0


### 2. Loading the Dataset with Pandas

Pandas provides tools for organizing and analyzing structured data. Before cleaning our dataset, we need to understand how its records are organized.

Our dataset contains product information, questions, and answers. We will first load Pandas and examine the structure of a sample record to determine how to transform the data into a table.

In [15]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

print("\nSample record:")
print(records[0].keys())

Pandas version: 3.1.0
NumPy version: 2.5.4

Sample record:
dict_keys(['asin', 'questions'])
